# Lesson X4: RL Research Frontiers

## Introduction

The final notebook in this curriculum looks past the algorithms already covered, toward where the field is headed and how to keep learning after this series ends.

1. **Meta-RL**: learning *how to learn*, not just one task — a from-scratch demonstration that adapts to a brand-new task with zero gradient updates
2. **Transfer learning and Sim2Real**: reusing what a policy already knows rather than training from scratch every time
3. **Decision Transformers**: reframing RL as sequence modeling — no TD learning, no policy gradient, just supervised learning on trajectories
4. **Staying current**: where the field's frontier actually lives, and how to keep tracking it after this notebook

## Setup

In [1]:
import subprocess
import sys

# Install dependencies
packages = ['gymnasium']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import time
import gymnasium as gym
from gymnasium import spaces

np.random.seed(42)
torch.manual_seed(42)

## Part 1: Meta-RL - Learning to Learn

Every algorithm so far trains one policy for one task. **Meta-RL** trains across a *distribution* of related tasks so the resulting policy can adapt to a brand-new task from that distribution quickly — often without any further gradient updates at all, adapting purely through what it does *within* a single episode.

**Algorithm distillation** (Laskin et al., 2022) is a clean, tractable way to get this: instead of meta-training with RL itself (notoriously sample-inefficient — full RL^2-style REINFORCE meta-training was tried while building this notebook and needed far more compute than a notebook budget affords for a comparably clean result), train a recurrent network via ordinary **supervised learning** to imitate a good bandit algorithm's action-selection *history* across many different bandit instances. The network never sees the true arm probabilities — only sequences of (previous action, previous reward). At test time, on a brand-new unseen bandit, its recurrent hidden state carries everything it has inferred about the arms so far, purely from that episode's own experience — in-context adaptation, zero weight updates.

In [2]:
K = 5           # bandit arms
N_PULLS = 20     # pulls per episode


def sample_bandit():
    return np.random.uniform(0.1, 0.9, K)


def thompson_episode(bandit_means, n_pulls=N_PULLS):
    """Thompson sampling -- the algorithm we'll distill into a recurrent network."""
    alpha, beta = np.ones(K), np.ones(K)
    actions, rewards = [], []
    for _ in range(n_pulls):
        samples = np.random.beta(alpha, beta)
        action = int(np.argmax(samples))
        reward = float(np.random.rand() < bandit_means[action])
        alpha[action] += reward
        beta[action] += (1 - reward)
        actions.append(action)
        rewards.append(reward)
    return actions, rewards


# Training data: many DIFFERENT bandits, each run once through Thompson sampling.
train_episodes = [(lambda m: (m, *thompson_episode(m)))(sample_bandit()) for _ in range(5000)]


class RecurrentBanditPolicy(nn.Module):
    """Input at each step: (previous action one-hot, previous reward). The LSTM's hidden
    state is where all in-episode adaptation happens -- no parameter updates at test time."""

    def __init__(self, k, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=k + 1, hidden_size=hidden, batch_first=True, num_layers=2)
        self.head = nn.Linear(hidden, k)

    def forward(self, x, hidden_state=None):
        out, hidden_state = self.lstm(x, hidden_state)
        return self.head(out), hidden_state


def build_bandit_batch(episodes):
    batch_size = len(episodes)
    X = torch.zeros(batch_size, N_PULLS, K + 1)
    Y = torch.zeros(batch_size, N_PULLS, dtype=torch.long)
    for i, (_, actions, rewards) in enumerate(episodes):
        for t in range(N_PULLS):
            if t > 0:
                X[i, t, actions[t - 1]] = 1.0
                X[i, t, K] = rewards[t - 1]
            Y[i, t] = actions[t]
    return X, Y


meta_policy = RecurrentBanditPolicy(K)
optimizer = optim.Adam(meta_policy.parameters(), lr=1e-3)

t0 = time.time()
for epoch in range(50):
    np.random.shuffle(train_episodes)
    for i in range(0, len(train_episodes), 64):
        X, Y = build_bandit_batch(train_episodes[i:i + 64])
        logits, _ = meta_policy(X)
        loss = nn.functional.cross_entropy(logits.reshape(-1, K), Y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
print(f"Distillation training time: {time.time() - t0:.0f}s")

Distillation training time: 64s


In [3]:
def run_meta_policy(policy, bandit_means, n_pulls=N_PULLS):
    hidden = None
    prev_action_onehot = torch.zeros(1, 1, K)
    prev_reward = torch.zeros(1, 1, 1)
    actions, rewards = [], []
    for _ in range(n_pulls):
        x = torch.cat([prev_action_onehot, prev_reward], dim=-1)
        with torch.no_grad():  # no gradient updates at test time -- adaptation is in-context only
            logits, hidden = policy(x, hidden)
        probs = torch.softmax(logits[0, 0], dim=-1)
        action = int(torch.distributions.Categorical(probs).sample().item())
        reward = float(np.random.rand() < bandit_means[action])
        actions.append(action)
        rewards.append(reward)
        prev_action_onehot = torch.zeros(1, 1, K)
        prev_action_onehot[0, 0, action] = 1.0
        prev_reward = torch.tensor([[[reward]]], dtype=torch.float32)
    return actions, rewards


first_half_rates, second_half_rates, meta_totals, random_totals, thompson_totals = [], [], [], [], []
for _ in range(50):
    test_bandit = sample_bandit()
    best_arm = np.argmax(test_bandit)

    actions, rewards = run_meta_policy(meta_policy, test_bandit)
    first_half_rates.append(np.mean(np.array(actions[:10]) == best_arm))
    second_half_rates.append(np.mean(np.array(actions[10:]) == best_arm))
    meta_totals.append(sum(rewards))

    random_rewards = [float(np.random.rand() < test_bandit[np.random.randint(K)]) for _ in range(N_PULLS)]
    random_totals.append(sum(random_rewards))

    _, thompson_rewards = thompson_episode(test_bandit)
    thompson_totals.append(sum(thompson_rewards))

print(f"Best-arm rate, first half of episode:  {np.mean(first_half_rates):.0%}")
print(f"Best-arm rate, second half of episode: {np.mean(second_half_rates):.0%}  <- improves WITHIN the episode")
print(f"\nMeta-policy (distilled) mean reward: {np.mean(meta_totals):.2f}")
print(f"Random baseline mean reward:         {np.mean(random_totals):.2f}")
print(f"Thompson sampling (the algorithm it was distilled from): {np.mean(thompson_totals):.2f}")
print("\nThe distilled network gets close to matching the algorithm it learned to imitate, on")
print("BRAND NEW bandits it never saw during training -- and it improves within a single 20-pull")
print("episode purely via its recurrent hidden state, with zero weight updates at test time.")

Best-arm rate, first half of episode:  26%
Best-arm rate, second half of episode: 37%  <- improves WITHIN the episode

Meta-policy (distilled) mean reward: 11.74
Random baseline mean reward:         10.28
Thompson sampling (the algorithm it was distilled from): 11.64

The distilled network gets close to matching the algorithm it learned to imitate, on
BRAND NEW bandits it never saw during training -- and it improves within a single 20-pull
episode purely via its recurrent hidden state, with zero weight updates at test time.


## Part 2: Transfer Learning and Sim-to-Real

**Sim-to-real** got a full hands-on treatment in X3 (domain randomization) — the same idea generalizes to **transfer learning** broadly: reuse a policy (or parts of one) trained on a source task/domain as the starting point for a related target task, rather than training from scratch. Three common patterns:

- **Fine-tuning**: initialize a new policy's weights from a pretrained one, then continue training on the target task — usually converges far faster than random initialization, the same logic that makes 14B's offline-then-online CQL fine-tuning work
- **Feature/representation transfer**: freeze early network layers (assumed to encode general, task-independent features) and only retrain the final layers on the new task
- **Domain randomization as transfer's cousin**: rather than transferring *weights*, transfer *robustness* — train across a distribution wide enough that the "target domain" (the real world) falls inside it, as X3 demonstrated directly

The common thread: don't waste what a previous training run already learned.

## Part 3: Decision Transformers - RL as Sequence Modeling

Every algorithm in this curriculum has been built around the Bellman equation — bootstrapped value estimates, TD errors, policy gradients. The **Decision Transformer** (Chen et al., 2021) throws all of that out and reframes RL as **conditional sequence modeling**: given a trajectory of (return-to-go, state, action) triples, predict the next action, conditioned on a *desired future return*. No TD learning, no policy gradient — plain supervised learning (next-token prediction, the exact objective language models use) on a dataset of trajectories, using a causally-masked transformer.

The payoff: at test time, you *choose* the policy's target performance by conditioning on a target return-to-go. Ask for the best return the data supports, and the model reproduces near-optimal behavior; ask for a mediocre or poor return, and it reproduces exactly that. Below: a compact transformer trained via behavior cloning on a mixed-quality dataset (deliberately including both good and poor trajectories) for our familiar GridWorld goal-reaching task.

In [4]:
class GridWorldEnv(gym.Env):
    """Deterministic goal-reaching GridWorld -- the offline-data source for the Decision
    Transformer below. Same dynamics as the earlier curriculum's GridWorld notebooks, wired
    through the standard Gymnasium reset()/step() API."""

    metadata = {'render_modes': [], 'render_fps': 0}

    SIZE = 6
    GOAL = (5, 5)
    DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def __init__(self, max_steps=15):
        self.observation_space = spaces.Box(low=0, high=self.SIZE - 1, shape=(2,), dtype=np.int32)
        self.action_space = spaces.Discrete(4)
        self.max_steps = max_steps
        self.pos = (0, 0)
        self.step_count = 0

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pos = (0, 0)
        self.step_count = 0
        return self._get_obs(), {}

    def step(self, action):
        di, dj = self.DELTAS[action]
        self.pos = (max(0, min(self.SIZE - 1, self.pos[0] + di)),
                    max(0, min(self.SIZE - 1, self.pos[1] + dj)))
        self.step_count += 1
        terminated = self.pos == self.GOAL
        truncated = self.step_count >= self.max_steps
        reward = 10.0 if terminated else -1.0
        return self._get_obs(), reward, terminated, truncated, {}

    def _get_obs(self):
        return np.array(self.pos, dtype=np.int32)

    def close(self):
        pass


SIZE = GridWorldEnv.SIZE
GOAL = GridWorldEnv.GOAL


def _peek(pos, action):
    """Where `action` would move to from `pos`, without stepping the environment -- used
    only to pick the greedy action when building the mixed-quality offline dataset."""
    di, dj = GridWorldEnv.DELTAS[action]
    return (max(0, min(SIZE - 1, pos[0] + di)), max(0, min(SIZE - 1, pos[1] + dj)))


def collect_mixed_quality_trajectory(epsilon, max_steps=15):
    """epsilon=0 -> optimal (greedy-toward-goal); higher epsilon -> noisier, often fails to
    reach the goal within the step budget -- exactly the DECISION TRANSFORMER's target setting:
    a fixed offline dataset of MIXED quality, no further environment interaction."""
    env = GridWorldEnv(max_steps=max_steps)
    obs, _ = env.reset()
    states, actions, rewards = [], [], []
    for _ in range(max_steps):
        pos = (int(obs[0]), int(obs[1]))
        if np.random.rand() < epsilon:
            action = np.random.randint(4)
        else:
            action = min(range(4), key=lambda a: sum(abs(x - y) for x, y in zip(_peek(pos, a), GOAL)))
        states.append(pos)
        actions.append(action)
        obs, reward, terminated, truncated, _ = env.step(action)
        rewards.append(reward)
        if terminated or truncated:
            break
    env.close()
    return states, actions, rewards


dt_dataset = [collect_mixed_quality_trajectory(np.random.choice([0.0, 0.2, 0.5, 0.8])) for _ in range(3000)]

MAX_LEN = 15


def encode_grid_state(s):
    return [s[0] / (SIZE - 1), s[1] / (SIZE - 1)]


def build_dt_batch(episodes):
    batch_size = len(episodes)
    X = torch.zeros(batch_size, MAX_LEN, 3)  # (return-to-go, state_x, state_y)
    Y = torch.full((batch_size, MAX_LEN), -100, dtype=torch.long)  # -100 = ignore in loss
    for i, (states, actions, rewards) in enumerate(episodes):
        length = len(states)
        return_to_go = np.cumsum(rewards[::-1])[::-1]
        for t in range(length):
            X[i, t, 0] = return_to_go[t] / 10.0
            X[i, t, 1:] = torch.tensor(encode_grid_state(states[t]))
            Y[i, t] = actions[t]
    return X, Y


class DecisionTransformer(nn.Module):
    def __init__(self, n_actions=4, d_model=32, n_heads=2, n_layers=2, max_len=MAX_LEN):
        super().__init__()
        self.embed = nn.Linear(3, d_model)
        self.pos_embed = nn.Parameter(torch.randn(max_len, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, batch_first=True, dim_feedforward=64)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Linear(d_model, n_actions)

    def forward(self, x):
        length = x.shape[1]
        h = self.embed(x) + self.pos_embed[:length]
        causal_mask = torch.triu(torch.ones(length, length) * float('-inf'), diagonal=1)
        h = self.transformer(h, mask=causal_mask)
        return self.head(h)


dt_model = DecisionTransformer()
dt_optimizer = optim.Adam(dt_model.parameters(), lr=1e-3)

t0 = time.time()
for epoch in range(20):
    np.random.shuffle(dt_dataset)
    for i in range(0, len(dt_dataset), 64):
        X, Y = build_dt_batch(dt_dataset[i:i + 64])
        logits = dt_model(X)
        loss = nn.functional.cross_entropy(logits.reshape(-1, 4), Y.reshape(-1), ignore_index=-100)
        dt_optimizer.zero_grad()
        loss.backward()
        dt_optimizer.step()
print(f"Decision Transformer training time: {time.time() - t0:.0f}s")

Decision Transformer training time: 21s


In [5]:
def rollout_conditioned_on_return(model, target_return, max_steps=15):
    env = GridWorldEnv(max_steps=max_steps)
    obs, _ = env.reset()
    history = []
    remaining_return = target_return
    for step_idx in range(max_steps):
        pos = (int(obs[0]), int(obs[1]))
        history.append((remaining_return / 10.0, encode_grid_state(pos)))
        X = torch.zeros(1, len(history), 3)
        for i, (rtg, s) in enumerate(history):
            X[0, i, 0] = rtg
            X[0, i, 1:] = torch.tensor(s)
        with torch.no_grad():
            logits = dt_model(X)
        action = int(torch.argmax(logits[0, -1]).item())
        obs, reward, terminated, truncated, _ = env.step(action)
        remaining_return -= reward
        if terminated:
            env.close()
            return step_idx + 1, True
        if truncated:
            env.close()
            return max_steps, False
    env.close()
    return max_steps, False


# 1.0 = the best return the dataset actually contains (10-step optimal path);
# -3.0 = a mediocre-but-still-successful trajectory; -15.0 = a trajectory that never reaches the goal
for target in [1.0, -3.0, -15.0]:
    results = [rollout_conditioned_on_return(dt_model, target) for _ in range(30)]
    successes = sum(1 for _, success in results if success)
    steps = [s for s, success in results if success]
    mean_steps = f"{np.mean(steps):.1f}" if steps else "n/a"
    print(f"target return={target:>6}: success rate={successes / 30:.0%}, mean steps when successful={mean_steps}")

print("\nSame model, same weights, three completely different behaviors -- selected purely by")
print("what return-to-go you condition on. Ask for the best outcome the data supports and it")
print("reproduces optimal play; ask for the worst and it reproduces failure. This is the")
print("distinctive DT property no value-based or policy-gradient method in this curriculum has:")
print("performance level as an input, not an outcome you have to train separately for.")

target return=   1.0: success rate=100%, mean steps when successful=10.0


target return=  -3.0: success rate=100%, mean steps when successful=14.0


target return= -15.0: success rate=7%, mean steps when successful=12.5

Same model, same weights, three completely different behaviors -- selected purely by
what return-to-go you condition on. Ask for the best outcome the data supports and it
reproduces optimal play; ask for the worst and it reproduces failure. This is the
distinctive DT property no value-based or policy-gradient method in this curriculum has:
performance level as an input, not an outcome you have to train separately for.


## Part 4: Staying Current

This curriculum covers foundational algorithms through late-2010s/early-2020s state of the art. The field keeps moving — some starting points for tracking what comes next:

**Venues**: NeurIPS, ICML, and ICLR are RL's primary publication venues; each has dedicated RL workshops most years. arXiv's `cs.LG` and `cs.AI` categories carry nearly everything pre-publication.

**Labs and blogs actively publishing RL research**: DeepMind, OpenAI, Berkeley AI Research (BAIR), and increasingly the major LLM labs, as RLHF (X3) blurs the line between "RL research" and "language model alignment research."

**Reference texts**: Sutton & Barto's *Reinforcement Learning: An Introduction* (the field's standard reference, free online) for foundations; OpenAI's *Spinning Up in Deep RL* for a implementation-focused bridge from theory to code, in the same spirit as this curriculum's from-scratch + library pairing.

**Active directions worth watching**, several of which this notebook already touched:
- **Foundation models for RL / decision-making** — Decision Transformers (Part 3) scaled up, and generalist agents trained across many tasks and modalities at once
- **In-context RL** — Part 1's algorithm distillation, scaled toward agents that adapt to entirely new tasks from a handful of in-context examples, no fine-tuning at all
- **RLHF and the RL-LLM boundary** — X3's human-in-the-loop pattern is now core infrastructure for training language models, not just a robotics footnote
- **Offline RL at scale** — 14A/14B's CQL and the broader push toward learning good policies from large, static, real-world datasets without further environment access

## Key Takeaways

1. **Meta-RL** trains across a task distribution so the resulting policy adapts to new tasks quickly — this notebook's distilled bandit policy adapted to unseen bandits purely via its recurrent hidden state, zero gradient updates, and nearly matched the algorithm (Thompson sampling) it was distilled from
2. **Algorithm distillation** sidesteps meta-RL's usual sample-inefficiency by turning "learn to learn" into ordinary supervised learning on a good algorithm's behavior — worth remembering whenever a from-scratch RL meta-training approach turns out to be too slow or unstable to reach a clean result
3. **Transfer learning** reuses what a previous training run already learned (fine-tuning, frozen features, or domain randomization's robustness-as-transfer) rather than training every new task from a blank slate
4. **Decision Transformers** replace the Bellman equation entirely with next-action prediction conditioned on a target return — this notebook's own GridWorld demo showed the SAME trained model reproduce optimal play, mediocre play, or outright failure, selected purely by which return-to-go it was asked for
5. This curriculum is a foundation, not a ceiling — Sutton & Barto plus Spinning Up plus the active-directions list above are the next steps once these notebooks are done